# 03 — Raw to Bronze (Delta Overwrite from Parquet)
Reads latest raw Parquet snapshots and writes exact Delta copies into Bronze.
**No transformations** — Bronze must be an identical Delta mirror of Raw.

In [0]:
# Widget parameters
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_name", "raw_zone", "Schema Name")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook", "Base Path")
dbutils.widgets.text("table_name", "all", "Table Name")
dbutils.widgets.text("run_id", "", "Run ID (auto-generated if blank)")

catalog = dbutils.widgets.get("catalog_name")
run_id_param = dbutils.widgets.get("run_id")

In [0]:
from datetime import datetime
import uuid

# Get run_id from upstream
try:
    run_id = dbutils.jobs.taskValues.get(taskKey="task_02_load_raw", key="run_id")
    ts_path = dbutils.jobs.taskValues.get(taskKey="task_02_load_raw", key="ts_path")
except:
    run_id = run_id_param if run_id_param else str(uuid.uuid4())[:8]
    ts_path = None

base_volume_path = f"/Volumes/{catalog}/raw_zone/chinook"
print(f"Run ID: {run_id}")

Run ID: c48d845c


## 1. Get Active Tables and Latest Raw Paths

In [0]:
df_metadata = spark.sql(f"""
    SELECT table_name
    FROM {catalog}.metadata.pipeline_parent_metadata
    WHERE active_flag = 'Y'
    ORDER BY load_sequence
""")

tables = [row.table_name for row in df_metadata.collect()]
print(f"Tables to process: {len(tables)}")

Tables to process: 11


## 2. Helper: Find Latest Raw File Path

In [0]:
def get_latest_raw_path(table_name):
    """Navigate the date-partitioned volume structure to find the most recent snapshot."""
    table_path = f"{base_volume_path}/{table_name}"

    # If we have ts_path from upstream, use it directly
    if ts_path:
        direct_path = f"{table_path}/{ts_path}/{table_name}.parquet"
        try:
            dbutils.fs.ls(direct_path)
            return direct_path
        except:
            pass

    # Otherwise, find the latest by traversing year/month/day/time
    try:
        years = sorted(dbutils.fs.ls(table_path), key=lambda x: x.name, reverse=True)
        for year_dir in years:
            if not year_dir.name.replace("/", "").isdigit():
                continue
            months = sorted(dbutils.fs.ls(year_dir.path), key=lambda x: x.name, reverse=True)
            for month_dir in months:
                days = sorted(dbutils.fs.ls(month_dir.path), key=lambda x: x.name, reverse=True)
                for day_dir in days:
                    times = sorted(dbutils.fs.ls(day_dir.path), key=lambda x: x.name, reverse=True)
                    for time_dir in times:
                        return time_dir.path + f"{table_name}.parquet"
    except Exception as e:
        print(f"  Could not find raw path for {table_name}: {e}")
        return None

    return None

## 3. Load Raw Parquet → Bronze Delta (Overwrite)

In [0]:
bronze_results = []

for table_name in tables:
    start_time = datetime.now()

    try:
        # Get latest raw file path
        raw_path = get_latest_raw_path(table_name)
        if raw_path is None:
            raise FileNotFoundError(f"No raw parquet found for {table_name}")

        print(f"\nProcessing: {table_name}")
        print(f"  Raw path: {raw_path}")

        # Read raw Parquet
        df_raw = spark.read.parquet(raw_path)
        raw_count = df_raw.count()

        # Write to Bronze as Delta (overwrite — current snapshot only)
        bronze_table = f"{catalog}.bronze.{table_name}"
        df_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(bronze_table)

        # Verify Bronze count
        bronze_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {bronze_table}").collect()[0].cnt

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        status = "success" if raw_count == bronze_count else "count_mismatch"
        symbol = "✓" if status == "success" else "⚠"
        print(f"  {symbol} raw={raw_count}, bronze={bronze_count} ({duration:.1f}s)")

        bronze_results.append({
            "table_name": table_name,
            "raw_count": raw_count,
            "bronze_count": bronze_count,
            "status": status,
            "duration": duration
        })

        # Log metrics
        spark.sql(f"""
            INSERT INTO {catalog}.metadata.pipeline_child_metrics
            VALUES (
                '{run_id}', '{table_name}', 'raw_to_bronze',
                current_timestamp(), '{status}',
                {raw_count}, {bronze_count},
                NULL, NULL,
                '{raw_path}', NULL,
                '{start_time}', '{end_time}', {duration},
                current_timestamp()
            )
        """)

    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_msg = str(e).replace("'", "''")[:500]
        print(f"  ✗ {table_name}: FAILED — {str(e)[:200]}")

        bronze_results.append({
            "table_name": table_name,
            "raw_count": 0,
            "bronze_count": 0,
            "status": "failed",
            "duration": duration
        })

        spark.sql(f"""
            INSERT INTO {catalog}.metadata.pipeline_child_metrics
            VALUES (
                '{run_id}', '{table_name}', 'raw_to_bronze',
                current_timestamp(), 'failed',
                0, 0,
                NULL, NULL, '', '{error_msg}',
                '{start_time}', '{end_time}', {duration},
                current_timestamp()
            )
        """)


Processing: album
  Raw path: dbfs:/Volumes/workspace/raw_zone/chinook/album/2026/03/28/053151/album.parquet
  ✓ raw=347, bronze=347 (7.4s)

Processing: artist
  Raw path: dbfs:/Volumes/workspace/raw_zone/chinook/artist/2026/03/28/053151/artist.parquet
  ✓ raw=275, bronze=275 (5.8s)

Processing: customer
  Raw path: dbfs:/Volumes/workspace/raw_zone/chinook/customer/2026/03/28/053151/customer.parquet
  ✓ raw=59, bronze=59 (6.4s)

Processing: employee
  Raw path: dbfs:/Volumes/workspace/raw_zone/chinook/employee/2026/03/28/053151/employee.parquet
  ✓ raw=8, bronze=8 (5.6s)

Processing: genre
  Raw path: dbfs:/Volumes/workspace/raw_zone/chinook/genre/2026/03/28/053151/genre.parquet
  ✓ raw=25, bronze=25 (7.0s)

Processing: invoice
  Raw path: dbfs:/Volumes/workspace/raw_zone/chinook/invoice/2026/03/28/053151/invoice.parquet
  ✓ raw=412, bronze=412 (5.8s)

Processing: invoiceline
  Raw path: dbfs:/Volumes/workspace/raw_zone/chinook/invoiceline/2026/03/28/053151/invoiceline.parquet
  ✓ raw

## 4. Bronze Load Summary

In [0]:
import pandas as pd

summary_df = pd.DataFrame(bronze_results)
print(f"\n{'='*60}")
print(f"RAW → BRONZE SUMMARY — Run ID: {run_id}")
print(f"{'='*60}")
print(f"Total:          {len(bronze_results)}")
print(f"Successful:     {sum(1 for r in bronze_results if r['status'] == 'success')}")
print(f"Mismatches:     {sum(1 for r in bronze_results if r['status'] == 'count_mismatch')}")
print(f"Failed:         {sum(1 for r in bronze_results if r['status'] == 'failed')}")
print(f"{'='*60}")

display(spark.createDataFrame(summary_df))


RAW → BRONZE SUMMARY — Run ID: c48d845c
Total:          11
Successful:     11
Mismatches:     0
Failed:         0


table_name,raw_count,bronze_count,status,duration
album,347,347,success,7.371369
artist,275,275,success,5.830712
customer,59,59,success,6.350319
employee,8,8,success,5.555288
genre,25,25,success,6.980686
invoice,412,412,success,5.757747
invoiceline,2240,2240,success,6.084658
mediatype,5,5,success,5.308194
playlist,18,18,success,6.049608
playlisttrack,8715,8715,success,5.481046


In [0]:
# Sample one Bronze table for verification
display(spark.sql(f"SELECT * FROM {catalog}.bronze.customer LIMIT 5"))

CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
2,Leonie,Köhler,null,Theodor-Heuss-Straße 34,Stuttgart,null,Germany,70174,+49 0711 2842222,null,leonekohler@surfeu.de,5
3,François,Tremblay,null,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,null,ftremblay@gmail.com,3
4,Bjørn,Hansen,null,Ullevålsveien 14,Oslo,null,Norway,0171,+47 22 44 22 22,null,bjorn.hansen@yahoo.no,4
5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,null,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


In [0]:
dbutils.jobs.taskValues.set(key="run_id", value=run_id)